# OpenAI Python SDK

Initialize the official client, send Chat Completions requests, interpret response fields, and handle auth, rate-limit, and network failures.


## 1. Overview

This guide covers:

- Bootstrapping `OpenAI()` with `python-dotenv` from the project root
- Chat Completions: `messages`, roles, `model`, `temperature`, `max_tokens`
- Reading `choices`, `finish_reason`, `usage`, and `id` from responses
- Typed exception handling for authentication, rate limits, and connectivity
- Optional streaming for incremental output
- A local message-assembly helper (no API) mirroring SDK patterns


## 2. Motivation

Most application code talks to OpenAI through the **Python SDK**, not raw HTTP. Consistent patterns for loading secrets, building `messages`, parsing responses, and surfacing errors keep notebooks and services debuggable.

Failures that look like "the model is bad" are often wrong kernel, missing `.env`, placeholder keys, or unhandled `RateLimitError`. Standardize the client bootstrap once and reuse it everywhere.


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **`OpenAI` client** | Official SDK entry point; reads `OPENAI_API_KEY` from the environment |
| **Chat Completions** | `client.chat.completions.create(...)` — multi-turn chat interface |
| **`messages`** | List of `{role, content}` dicts: `system`, `user`, `assistant` |
| **`model`** | Model id (e.g. `gpt-4o-mini`) |
| **`temperature`** | Sampling randomness for completions |
| **`finish_reason`** | Why generation stopped: `stop`, `length`, `content_filter`, etc. |
| **`usage`** | Token counts: `prompt_tokens`, `completion_tokens`, `total_tokens` |
| **Streaming** | `stream=True` yields partial deltas as tokens arrive |

### 3.2 How it works

1. `load_dotenv(project_root / ".env")` injects secrets into `os.environ`.
2. `OpenAI()` constructs a client that attaches the API key to each request.
3. You pass `messages` and parameters to `chat.completions.create`.
4. The SDK serializes JSON, calls the REST API, and returns a typed response object.
5. Read `response.choices[0].message.content` for the assistant text.

### 3.3 When to use this pattern

**Use for:** application backends, notebooks, batch jobs, and prototypes calling OpenAI chat models.

**Trade-offs:** simple and well-documented; vendor-specific. For multi-provider apps, wrap the client behind an interface.

**Alternatives:** REST with `httpx`, LangChain/LlamaIndex abstractions, or Azure OpenAI with `AzureOpenAI` when hosted on Azure.


## 4. Architecture

```mermaid
flowchart TD
    env[.env OPENAI_API_KEY] --> dotenv[load_dotenv]
    dotenv --> client[OpenAI client]
    client --> req[chat.completions.create]
    req --> api[OpenAI REST API]
    api --> resp[ChatCompletion response]
    resp --> text[choices message content]
    resp --> meta[usage finish_reason id]
    req -.->|stream=True| stream[delta chunks]
```

### Message flow

```text
system  → behavior, policy, persona
user    → task and data
assistant → prior model turns (multi-turn history)
```


## 5. Local Python Examples


Build and validate `messages` locally before spending API credits.


In [1]:
# Local Chat Completions message builder — no API required
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal

Role = Literal["system", "user", "assistant"]


@dataclass
class ChatTurn:
    role: Role
    content: str


@dataclass
class MessageBuilder:
    # Assemble OpenAI-style messages with basic validation.

    turns: list[ChatTurn] = field(default_factory=list)

    def system(self, content: str) -> "MessageBuilder":
        self.turns.append(ChatTurn("system", content.strip()))
        return self

    def user(self, content: str) -> "MessageBuilder":
        self.turns.append(ChatTurn("user", content.strip()))
        return self

    def assistant(self, content: str) -> "MessageBuilder":
        self.turns.append(ChatTurn("assistant", content.strip()))
        return self

    def build(self) -> list[dict[str, str]]:
        if not any(t.role == "user" for t in self.turns):
            raise ValueError("At least one user message is required.")
        return [{"role": t.role, "content": t.content} for t in self.turns]

    def token_estimate_chars(self) -> int:
        # Rough char count; use tiktoken for real estimates.
        return sum(len(t.content) for t in self.turns)


messages = (
    MessageBuilder()
    .system("You are a concise SRE assistant.")
    .user("Summarize why health checks should avoid live API calls.")
    .build()
)

print("Built messages:")
for m in messages:
    preview = m["content"][:60].replace("\n", " ")
    print(f"  [{m['role']}] {preview}...")
print(f"Approx chars: {sum(len(m['content']) for m in messages)}")


Built messages:
  [system] You are a concise SRE assistant....
  [user] Summarize why health checks should avoid live API calls....
Approx chars: 88


## 6. OpenAI SDK Examples

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(root / ".env")
client = OpenAI()
```


In [2]:
# Client bootstrap and connectivity
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import (
    APIConnectionError,
    AuthenticationError,
    OpenAI,
    RateLimitError,
)

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )

root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()
MODEL = "gpt-4o-mini"


def api_key_ready() -> bool:
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip():
        print("OPENAI_API_KEY not configured. Copy .env.example to .env and set your key.")
        return False
    if "your_openai_api_key" in key.lower():
        print("OPENAI_API_KEY is still a placeholder.")
        return False
    return True


def run_connectivity_check() -> None:
    if not api_key_ready():
        return

    messages = [
        {"role": "system", "content": "You are a concise assistant. Reply in one short sentence."},
        {"role": "user", "content": "Say: SDK connectivity OK."},
    ]
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0,
            max_tokens=20,
        )
        print("API call succeeded.")
        print(f"id    : {response.id}")
        print(f"model : {response.model}")
        print(f"reply : {response.choices[0].message.content}")
        if response.usage:
            print(
                f"tokens: prompt={response.usage.prompt_tokens}, "
                f"completion={response.usage.completion_tokens}"
            )
    except AuthenticationError:
        print("Authentication failed. Check OPENAI_API_KEY.")
    except RateLimitError:
        print("Rate limit or quota hit. Wait and retry.")
    except APIConnectionError as exc:
        print(f"Network error: {exc}")


run_connectivity_check()


API call succeeded.
id    : chatcmpl-EAHKaWLLv50hpYSDY7rhIoeyBcq3b
model : gpt-4o-mini-2024-07-18
reply : SDK connectivity OK.
tokens: prompt=29, completion=4


### 6.1 Inspect full response fields


In [3]:
# Print structured response metadata
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIConnectionError, AuthenticationError, OpenAI, RateLimitError

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )

root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()
MODEL = "gpt-4o-mini"


def api_key_ready() -> bool:
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip() or "your_openai_api_key" in key.lower():
        print("OPENAI_API_KEY not configured or still placeholder.")
        return False
    return True


def inspect_response(user_text: str) -> None:
    if not api_key_ready():
        return

    messages = [
        {"role": "system", "content": "Reply with valid JSON only."},
        {"role": "user", "content": user_text},
    ]
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.1,
            max_tokens=80,
        )
    except (AuthenticationError, RateLimitError, APIConnectionError) as exc:
        print(f"Request failed: {type(exc).__name__}: {exc}")
        return

    choice = response.choices[0]
    print("--- choice ---")
    print(f"index         : {choice.index}")
    print(f"finish_reason : {choice.finish_reason}")
    print(f"role          : {choice.message.role}")
    print(f"content       : {choice.message.content}")
    print("--- usage ---")
    if response.usage:
        print(json.dumps(response.usage.model_dump(), indent=2))
    print(f"response.id   : {response.id}")


inspect_response('Return JSON: {"status": "ok", "items": ["a","b"]}')


--- choice ---
index         : 0
finish_reason : stop
role          : assistant
content       : {
  "status": "ok",
  "items": [
    "a",
    "b"
  ]
}
--- usage ---
{
  "completion_tokens": 24,
  "prompt_tokens": 35,
  "total_tokens": 59,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 0,
    "rejected_prediction_tokens": 0
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cache_write_tokens": null,
    "cached_tokens": 0
  }
}
response.id   : chatcmpl-EAHKb9uJhR18Rf8WHPBL6fKKYwUo8


### 6.2 Streaming (optional)


In [4]:
# Stream tokens as they arrive — brief demo
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIConnectionError, AuthenticationError, OpenAI, RateLimitError

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )

root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()
MODEL = "gpt-4o-mini"


def api_key_ready() -> bool:
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip() or "your_openai_api_key" in key.lower():
        print("OPENAI_API_KEY not configured or still placeholder.")
        return False
    return True


def stream_reply(prompt: str) -> None:
    if not api_key_ready():
        return

    messages = [
        {"role": "system", "content": "Be brief."},
        {"role": "user", "content": prompt},
    ]
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.2,
            max_tokens=60,
            stream=True,
        )
        print("Stream: ", end="", flush=True)
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            print(delta, end="", flush=True)
        print()
    except (AuthenticationError, RateLimitError, APIConnectionError) as exc:
        print(f"Stream failed: {type(exc).__name__}: {exc}")


stream_reply("Name three HTTP status codes and one word each.")


Stream: 

1

.

200

 -

...

3

.

500

 -

 Error

## 7. Implementation notes

1. **Load order** — Call `load_dotenv(root / ".env")` before `OpenAI()` so the key is present at client construction.
2. **`api_key_ready`** — Central guard avoids stack traces when `.env` is missing; print actionable messages instead.
3. **Typed exceptions** — Catch `AuthenticationError`, `RateLimitError`, and `APIConnectionError` separately for ops playbooks.
4. **`finish_reason=length`** — Output was cut by `max_tokens`; increase limit or shorten the prompt.
5. **Streaming** — Use for UX latency; aggregate deltas server-side if you need the full string for parsing.
6. **`MessageBuilder`** — Local validation catches empty user messages before a network round trip.


## 8. Best practices

- Never hardcode API keys; load from `.env` or a secret manager.
- Pin `model` in config; log `response.model` to detect silent routing changes.
- Set `temperature=0` (or low) for structured extraction; raise slightly for creative drafts.
- Log `usage` on every call for cost dashboards.
- Use `max_tokens` to cap runaway completions and cost.
- Wrap SDK calls in a thin service function so notebooks and apps share one error-handling path.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `AuthenticationError` | Missing, placeholder, or revoked key | Set real `OPENAI_API_KEY` in project-root `.env` |
| `RateLimitError` | Quota, concurrency, or TPM limits | Backoff, batch, or upgrade plan |
| `APIConnectionError` | Network, proxy, or DNS | Check connectivity; retry with timeout |
| Empty `content` | Refusal, filter, or tool-call response | Inspect `finish_reason` and message object |
| `ModuleNotFoundError: openai` | Wrong Jupyter kernel | Select project `.venv` kernel |
| Key works in terminal, not notebook | cwd / `.env` path mismatch | Use `find_project_root()` + explicit `.env` path |
| Huge bills | No `max_tokens`, long prompts | Cap tokens; measure with `tiktoken` first |


## 10. Validation checklist

1. Run the message builder cell; confirm at least one `user` role is required.
2. Run connectivity check with a valid key; confirm `id`, `model`, and reply print.
3. Run response inspection; confirm `finish_reason` and `usage` fields appear.
4. Optionally run streaming; confirm incremental output without errors.
5. Temporarily break the key in `.env` and confirm graceful skip (no uncaught exception).


## 11. Summary

- Bootstrap with `load_dotenv(root / ".env")` and `OpenAI()`.
- Chat Completions take a `messages` list with `system` / `user` / `assistant` roles.
- Parse `choices[0].message.content`, `finish_reason`, and `usage` for every call.
- Handle auth, rate-limit, and network errors explicitly.

**Next:** `Anatomy_of_a_Great_Prompt.ipynb` — decompose prompts into role, task, context, constraints, and format.
